# Agentic AI: step by step

**Part A: Setup.** Run these cells first in every new Colab session. Colab forgets everything when a session ends, so we rebuild the setup each time.



```powershell
cd D:\parham\Parham_Learning\agentic-ai
Rename-Item stage-00-colab-check.ipynb stage-00-colab-scratch.ipynb
Set-Content -Path 01-agent-basics.ipynb -Value '{"cells":[],"metadata":{},"nbformat":4,"nbformat_minor":5}'
code 01-agent-basics.ipynb
```

### Cell 1: Check the GPU
Confirms Colab gave us a GPU. We want `Tesla T4`.

In [14]:
# Cell 1: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### Cell 2: Install helper tools
`zstd` unpacks Ollama's files. `pciutils` and `lshw` let the installer find the GPU. We install them first so the GPU is detected on the first try.

In [15]:
# Cell 2: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


### Cell 3: Install Ollama
Installs the Ollama program. Look for "NVIDIA GPU installed" near the end. The "systemd is not running" warning is normal in Colab.

In [16]:
# Cell 3: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


### Cell 4: Start the server
The server is the program that waits for requests. It must be running before Python can talk to the model. We run it in the background and save its messages to `ollama.log`.

In [17]:
# Cell 4: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

### Cell 5: Check the server found the GPU
Searches the server log for GPU lines. We want `Tesla T4` and about 14 GiB available.

In [18]:
# Cell 5: Check the server log for GPU detection
!grep -i -E "gpu|cuda|nvidia" ollama.log | head -n 10

time=2026-10-15T21:41:20.248Z level=INFO source=routes.go:2439 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https:

### Cell 6: Install the Python client
The `ollama` package lets Python send messages to the server.

In [19]:
# Cell 6: Install the Ollama Python client
!pip install -q ollama

### Cell 7: Download the model
Downloads Gemma 4 (small size) onto Colab's disk. We use Python so the output stays clean. A reply of `success` means it worked.

In [20]:
# Cell 7: Download the model through the Python client
import ollama

print(ollama.pull("gemma4:e2b").status)

success


### Cell 8: First message
A quick test that the whole setup works. The first call is slow because the model loads onto the GPU.

In [21]:
# Cell 8: Send one message and print the reply
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)
print(response.message.content)

Hello, I hope you are having a wonderful day.


## Part B: Stage 1, talking to the model from Python

The call stays the same. Only the message text changes. We try four different messages and compare.

### Cell 9: Ask a plain question
The simplest use. We send one question and print the reply.

In [ ]:
# Cell 9: Ask a plain question
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "What is a Bill of Materials (BOM) in two sentences?"}],
)

print(response.message.content)

A Bill of Materials (BOM) is a comprehensive list that details all the raw materials, components, and sub-assemblies required to manufacture a specific product. It is crucial for inventory management, cost estimation, and ensuring that all necessary parts are accounted for throughout the production process.


### Cell 10: Control the length
The model follows instructions about size. We ask for exactly one sentence, then compare with Cell 9.

In [23]:
# Cell 10: Ask for a very short answer

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "In one short sentence, what is a Bill of Materials (BOM)?"}],)
print(response.message.content)

A Bill of Materials (BOM) is a comprehensive list of all the raw materials, components, and sub-assemblies required to manufacture a product.


### Cell 11: Control the style
We explain the same idea to a different reader. The facts stay the same, and the wording changes.

In [24]:
# Cell 11: Explain the same idea to a child

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Explain what a Bill of Materials (BOM) is to a child."}],)
print(response.message.content)

Imagine you are going to build the coolest LEGO spaceship ever!

You need lots of different pieces: big blue bricks, small red wheels, clear windows, and maybe some special stickers.

If you just start building without a plan, you might forget a piece, or maybe you end up with a spaceship that is missing a wing!

### What is a Bill of Materials (BOM)?

A **Bill of Materials** is like the **official shopping list** or the **secret instruction book** for building anything big.

It is a very, very important list that tells everyone exactly what things you need to make a finished product.

**Here is how it works:**

1. **The Product:** Let's say the finished product is a cool robot toy.
2. **The List (The BOM):** The Bill of Materials is the list that says:
    * 5 wheels
    * 1 big body
    * 3 arms
    * 1 battery
    * 1 special head
3. **The Purpose:** It makes sure that the people building the robot don't forget anything and that they have *exactly* the right amount of every piece.



### Cell 12: Control the format
We ask for a table. Later, the same idea will let us ask for structured data that code can read.

In [25]:
# Cell 12: Ask for a markdown table
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Make a markdown table of 3 parts in a simple desk lamp: part name, material, and quantity."}],
)
print(response.message.content)

| Part Name | Material | Quantity |
| :--- | :--- | :--- |
| Lamp Base/Stand | Metal (e.g., Steel or Aluminum) | 1 |
| Lamp Shade | Fabric (e.g., Cotton or Linen) | 1 |
| Electrical Cord | Copper Wire (with PVC insulation) | 1 |
| Switch Mechanism | Plastic or Metal | 1 |
| Mounting Hardware | Screws/Bolts | 4-6 |


### Cell 13: Improve a message
Earlier, the model skipped a pronunciation. A clearer instruction gives a more complete answer.

In [26]:
# Cell 13: Ask for hello in four languages with a pronunciation for every word

response = ollama.chat(
    model="gemma4:e2b",
    messages = [{"role": "user", "content": (
        "Say hello in Georgian, Farsi, Welsh and Japanese. "
        "Use one greeting per language. "
        "Give the pronunciation of every greeting in parentheses."
    )}]
)
print(response.message.content)


Here are greetings in the requested languages:

**Georgian:** Gamarjoba (gah-mar-jo-ba)

**Farsi:** Salaam (sa-laam)

**Welsh:** Hello (hel-loh)

**Japanese:** Konnichiwa (koh-nee-chee-wah)


## Stage 1 result: the message controls the answer

| Cell | What we changed | Result |
|---|---|---|
| 9 | A plain question | A normal answer |
| 10 | Asked for one short sentence | A shorter answer |
| 11 | Asked for a child's level | Simpler wording |
| 12 | Asked for a table | A table |
| 13 | Added clearer rules | A more complete answer |

**Lesson:** the model does what the message says. Clear instructions give better answers.